# boolean-mask-identity-replace — ex3: zero out flagged rows of a 2-D tensor

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `boolean-mask-identity-replace`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five mask-and-substitute patterns that ramp from `x < 0` → clamp → row-zero → identity-substitute → safe batched solve. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `boolean-mask-identity-replace`**, which bridges to the bank subtopic `Numpy: Indexing and selection` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "boolean-mask-identity-replace"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Mask & substitute — quick refresher

**Build a mask.** Any comparison returns a `dtype=bool` tensor with the same shape as its input: `x < 0`, `x.abs() < eps`, `(x > 0) & (x < 1)`.

**Write through a mask.** `y[mask] = value` modifies in place. If `value` is a scalar, it broadcasts over the masked region. If `value` is a tensor, its shape must match the shape of `y[mask]` after broadcasting.

**Always clone first** if you want a non-mutating function — `y = x.clone(); y[mask] = 0; return y`. Otherwise the caller's input gets clobbered.

**Identity substitute.** `A[singular_mask] = torch.eye(N)` replaces flagged `(N, N)` submatrices with the identity — the standard cleanup before a batched `linalg.solve` so one degenerate slice doesn't crash the whole batch.

### Exercise 3 — zero out flagged rows of a 2-D tensor

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply a 1-D bool mask on the first axis of a 2-D tensor to zero out whole rows in one assignment.
> Keywords: row-mask, broadcast-assign, batch-zero
> ```

**KCs targeted:** `row-mask-broadcast-assign`

Implement `ex3_zero_flagged_rows(b, mask)`. Given a `(B, N)` tensor `b` and a 1-D bool mask of shape `(B,)`, return a copy of `b` where every row flagged `True` has been set to all zeros. Unflagged rows stay unchanged.

Strategy: `out = b.clone(); out[mask] = 0.0` — the scalar `0.0` broadcasts to fill every selected row.

Input `b` must NOT be mutated.

In [ ]:
def ex3_zero_flagged_rows(b: Tensor, mask: Tensor) -> Tensor:
    """Zero out rows of b where mask is True. Does NOT mutate b."""
    raise NotImplementedError()


def _test_ex3():
    b = t.tensor([
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
        [7.0, 8.0, 9.0],
        [10.0, 11.0, 12.0],
    ])
    b_orig = b.clone()
    mask = t.tensor([False, True, False, True])
    out = ex3_zero_flagged_rows(b, mask)
    assert out.shape == b.shape, f'shape mismatch: {out.shape} vs {b.shape}'
    expected = t.tensor([
        [1.0, 2.0, 3.0],
        [0.0, 0.0, 0.0],
        [7.0, 8.0, 9.0],
        [0.0, 0.0, 0.0],
    ])
    assert t.allclose(out, expected), f'value mismatch:\n{out}'
    assert t.allclose(b, b_orig), 'input tensor was mutated'
    # All-False mask should leave b unchanged.
    no_mask = t.zeros(4, dtype=t.bool)
    assert t.allclose(ex3_zero_flagged_rows(b, no_mask), b), 'all-False mask should be no-op'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_zero_flagged_rows(b: Tensor, mask: Tensor) -> Tensor:
    out = b.clone()
    out[mask] = 0.0
    return out
```

**Why this works.** Indexing `out[mask]` with a 1-D bool mask of shape `(B,)` against a `(B, N)` tensor selects `K = mask.sum()` rows, giving a `(K, N)` view. Writing scalar `0.0` broadcasts across that view to fill every selected row with zeros.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()